# 02 · Where the delay minutes come from

**RQ1.** How are reported delay minutes distributed across the five BTS causes, and
how stable is that composition across the panel?

The pilot year gave a 75/25 split between airline-controlled and system-imposed
minutes. A single year cannot separate a structure from a one-year shock, so the
split is reported with a bootstrap interval and recomputed per year.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from team01 import config, io, clean

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
print(config.summarise())

Team01-FlightDelay-Project v0.2.0 | mode=sample | years=2015-2024 | flights=flights_sample.parquet


In [2]:
from team01 import attribution, viz

raw = io.load_flights()
panel = clean.completed(raw)
universe = clean.attribution_universe(raw)

totals = attribution.cause_totals(universe)
totals[["cause", "family", "minutes", "share"]]

,cause,family,minutes,share
0,Carrier,Internal,235222.437500,0.348026
1,Late aircraft,Internal,220289.000000,0.325931
2,Air traffic / NAS,System,132248.812500,0.195670
3,Extreme weather,System,79465.132812,0.117573
4,Security,System,8651.371094,0.012800


## 2.1 The headline split, with an interval

The resampling unit is the **operating day**, not the flight: flights delayed by one
storm are not independent, so a flight-level bootstrap would produce an interval far
too narrow.

In [3]:
split = attribution.family_split(universe)
interval = attribution.bootstrap_family_share(universe)

print(f"internal share  {interval['share']:.3f}  "
      f"[{interval['ci_low']:.3f}, {interval['ci_high']:.3f}]  "
      f"over {interval['days']} operating days")
print(f"late aircraft   {split['late_aircraft_share']:.1%} of attributed minutes")

internal share  0.674  [0.670, 0.677]  over 120 operating days
late aircraft   32.6% of attributed minutes


## 2.2 Is the composition stable?

Ten years of a small move is the interesting question: if the interval for one year
clears the interval for another, that is a structural break rather than noise.

In [4]:
attribution.stability_by_year(universe, bootstrap=True, reps=200)

,year,attributed_minutes,internal_share,system_share,late_aircraft_share,ci_low,ci_high
0,2019,226228.5,0.6693,0.3307,0.3223,0.6644,0.6750
1,2023,224156.6,0.6774,0.3226,0.3303,0.6703,0.6844
2,2024,225491.7,0.6753,0.3247,0.3252,0.6705,0.6819


## 2.3 The figure

A unit waffle, because the question is part-to-whole and the cells can be labelled
individually. Area is a weak channel, so the shares are written in the legend rather
than read off the shape.

In [5]:
viz.fig_cause_waffle(totals)

[viz] 01_cause_waffle.png


'C:\\Users\\lenovo\\Desktop\\Semester Project\\Team01-FlightDelay-Project\\figures\\01_cause_waffle.png'